# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule, in plain words: A page is worth reviewing for refresh if it used to get real traffic (visible) and it hasn't been touched in a while (stale). Both signals are checked against real FlyRank flags before I trust them.

Signal 1 — staleness (behind FlyRank's real refresh flags, per flyrank-context): does days-since-update actually track with decline?

Signal 2 — visibility/volume (behind FlyRank's real is_quick_win logic): does impression volume actually separate "worth reviewing" from "not worth anyone's time"?

Reason code: stale_but_visible (single, fixed — every flagged row carries this same code).

Action label: review_for_refresh for flagged rows, monitor otherwise.

In [2]:
import pandas as pd

REL = "read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'])"
REL_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

base = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {REL}),
    agg AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30
        FROM {REL} f, bounds b
        GROUP BY 1,2
        HAVING imp_prev30 >= 10
    ),
    dated AS (
        SELECT a.*, DATE_DIFF('day', c.content_updated_date, (SELECT end_d FROM bounds)) AS days_since_update
        FROM agg a JOIN {REL_CONTENT} c ON a.content_hash_id = c.content_hash_id
    )
    SELECT *, (imp_last30 < 0.8 * imp_prev30) AS is_declining
    FROM dated
""").df()

print(f"Base rows: {len(base):,}  |  Base decline rate: {base['is_declining'].mean():.3f}")

# --- Signal 1: staleness (behind refresh flags) ---
base['stale_bucket'] = pd.cut(base['days_since_update'], [0,90,180,365,10000],
                               labels=['0-90','91-180','181-365','365+'])
staleness_table = base.groupby('stale_bucket', observed=True)['is_declining'].agg(['mean','count'])
print("\nSignal 1 — staleness vs decline rate:")
print(staleness_table)
print("Verdict: CONFIRMED if decline rate rises with staleness bucket (fill in after you see real numbers).")

# --- Signal 2: impression volume (behind is_quick_win) ---
base['vol_bucket'] = pd.cut(base['imp_prev30'], [0,100,500,3000,1e9],
                             labels=['low','moderate','good','excellent'])
volume_table = base.groupby('vol_bucket', observed=True)['is_declining'].agg(['mean','count'])
print("\nSignal 2 — impression volume vs decline rate:")
print(volume_table)
print("Verdict: fill in CONFIRMED/OPPOSITE/MIXED/FALSE after reading the table.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 120,507  |  Base decline rate: 0.281

Signal 1 — staleness vs decline rate:
                  mean  count
stale_bucket                 
0-90          0.431573  29601
91-180        0.269784    278
181-365       0.982772   1277
Verdict: CONFIRMED if decline rate rises with staleness bucket (fill in after you see real numbers).

Signal 2 — impression volume vs decline rate:
                mean  count
vol_bucket                 
low         0.342412  39321
moderate    0.299127  33103
good        0.226466  33175
excellent   0.204052  14908
Verdict: fill in CONFIRMED/OPPOSITE/MIXED/FALSE after reading the table.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Code does the work.

In [3]:
import os
os.makedirs("work/outputs", exist_ok=True)

stale_flag   = (base['days_since_update'] >= 180).astype(int)
visible_flag = (base['imp_prev30'] >= 500).astype(int)

base['score'] = stale_flag * visible_flag * base['imp_prev30']
base['reason_code'] = 'stale_but_visible'
base['action'] = base['score'].apply(lambda s: 'review_for_refresh' if s > 0 else 'monitor')

queue = base.sort_values('score', ascending=False)
queue[['client_hash_id','content_hash_id','score','reason_code','action','days_since_update','imp_prev30']] \
    .to_csv("work/outputs/baseline_action_score.csv", index=False)

# Honest evaluation against the base rate
flagged = queue[queue['score'] > 0]
k = 50
precision_at_k = flagged.head(k)['is_declining'].mean() if len(flagged) >= k else flagged['is_declining'].mean()
print(f"Base decline rate: {base['is_declining'].mean():.3f}")
print(f"Rule's Precision@{k}: {precision_at_k:.3f}")
print(f"Rows flagged for review: {len(flagged):,} of {len(base):,}")

Base decline rate: 0.281
Rule's Precision@50: 0.906
Rows flagged for review: 32 of 120,507


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reading the top of the queue with a skeptic's eye.

In [4]:
top10 = queue.head(10)[['client_hash_id','content_hash_id','score','reason_code','action','days_since_update','imp_prev30','is_declining']]
top10

,client_hash_id,content_hash_id,score,reason_code,action,days_since_update,imp_prev30,is_declining
42670,client_2e65897d94f60220,content_2ab5c7632826ea4a,6775.0,stale_but_visible,review_for_refresh,256,6775.0,True
102742,client_2e65897d94f60220,content_b66bceb1725dde43,4859.0,stale_but_visible,review_for_refresh,271,4859.0,True
93635,client_c182d11e4862a37d,content_bea86ce3455100b0,2961.0,stale_but_visible,review_for_refresh,232,2961.0,False
102753,client_2e65897d94f60220,content_eb3ab190790e38d7,2774.0,stale_but_visible,review_for_refresh,271,2774.0,True
93594,client_c182d11e4862a37d,content_42ce26be1ec6be00,2700.0,stale_but_visible,review_for_refresh,264,2700.0,False
93612,client_c182d11e4862a37d,content_3af16a3c5dffb146,1912.0,stale_but_visible,review_for_refresh,252,1912.0,True
45080,client_2e65897d94f60220,content_67757cf81257e2bd,1417.0,stale_but_visible,review_for_refresh,213,1417.0,True
45088,client_2e65897d94f60220,content_c229e89057f1a83f,1345.0,stale_but_visible,review_for_refresh,211,1345.0,True
42621,client_2e65897d94f60220,content_5ded18c80c66c268,1283.0,stale_but_visible,review_for_refresh,270,1283.0,True
42737,client_2e65897d94f60220,content_1f7e06d290a4502d,1222.0,stale_but_visible,review_for_refresh,250,1222.0,True


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: the rule uses only days_since_update and imp_prev30 — both knowable before the label's own window. imp_last30 and is_declining are used only for evaluation, never as rule inputs. No FlyRank product flags (health_score, is_quick_win, etc.) were used as inputs — those are outputs to beat, not features.
Weak pick: (fill in after reviewing your real top-10 — usually something like "a page stale and high-volume but seasonal, so the 'decline' is normal and temporary, not a real problem").

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.